In [0]:
TABLE_CUSTOMER_BRONZE = "customer_360.bronze.customers"
TABLE_SILVER_CUSTOMER = "customer_360.silver.customers"
TABLE_QUARANTINE_CUSTOMER = "customer_360.quarantine.customers"
TABLE_CUSTOMER_VIEWS="customer_360.bronze.customer_views"
CUSTOMER_METRICS_TABLE="customer_360.raw.customer_silver_metrics"
PATH_CUSTOMER_CHECKPOINTLOCATION_SILVER = (
    "/Volumes/customer_360/raw/source_files/checkpoints/silver/customers"
)

TABLE_METRIC = "customer_360.raw.stream_metrics"

In [0]:
# Customer Silver Table
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE_SILVER_CUSTOMER} (
    customer_id STRING NOT NULL,
    first_name STRING,
    last_name STRING,
    email STRING,
    phone STRING,
    city STRING,
    region STRING,
    customer_segment STRING,
    registration_date DATE,
    updated_at TIMESTAMP
)
USING DELTA
""")

# Customer Quarantine Table
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE_QUARANTINE_CUSTOMER} (
    customer_id STRING,
    first_name STRING,
    last_name STRING,
    email STRING,
    phone STRING,
    city STRING,
    region STRING,
    customer_segment STRING,
    registration_date DATE,
    updated_at TIMESTAMP,
    failure_reason STRING,
    quarantined_at TIMESTAMP
)
USING DELTA
""")

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {TABLE_CUSTOMER_VIEWS} (
    customer_id STRING NOT NULL,
    updated_at TIMESTAMP,
    viewed_at TIMESTAMP
)
USING DELTA;
    """
)

spark.sql(f"""
CREATE TABLE IF NOT EXISTS  {CUSTOMER_METRICS_TABLE}(
    metric_time TIMESTAMP,
    batch_id BIGINT,
    query_name STRING,
    total_records BIGINT,
    valid_records BIGINT,
    invalid_records BIGINT,
    duplicate_records BIGINT
)
USING DELTA
""")

In [0]:
customer_df=(
    spark
    .readStream
    .format("delta")
    .table(TABLE_CUSTOMER_BRONZE)
)
# customer_df.display()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql import Row
from datetime import datetime
def process_dataframe(batch_df,batch_id):
    df=(
        batch_df
        .withColumn("email",trim(col("email")))
        .withColumn(
            "failure_reason",
            when(col("customer_id").isNull(),"customer_id is null")
            .when(col("first_name").isNull(),"first_name is null")
            .when(col("last_name").isNull(),"last_name is null")
            .when(col("email").isNull(),"email is null")
            .when(col("email").rlike(r"^[a-zA-Z0-9._%+-]+/+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$")
                  ,"email is not in proper format")
            .when(col("phone").isNull(),"phone is null")
            .when(col("city").isNull(),"city is null")
            .when(col("region").isNull(),"region is null")
            .when(col("customer_segment").isNull(),"customer_segment is null")
            .when(col("registration_date").isNull(),"registration_date is null")
            .when(col("updated_at").isNull(),"updated_at is null")
            .otherwise(None)
        )
    )

    # Filter clean data
    valid_data=(
        df
        .filter(col("failure_reason").isNull())
        .drop("failure_reason")
    )


    # Load quarantine data to quarantine db
    quarantine_data=(
        df
        .filter(col("failure_reason").isNotNull())
        .withColumn("quarantined_at",current_timestamp())
    )
    quarantine_data.write.mode("append").format("delta").saveAsTable(TABLE_QUARANTINE_CUSTOMER)
    
    window=Window.partitionBy(["customer_id","updated_at"]).orderBy(col("updated_at").asc())
    # Filter duplicate data
    valid_data=valid_data.withColumn(
        "rn",
        row_number().over(window)
    )
    unique_data=valid_data.filter(col("rn")==1).drop("rn")
    duplicate_data=valid_data.filter(col("rn")>1).drop("rn")

    # Filter duplicate from seen data 
    # Load seen data
    first_occurance=spark.read.format("delta").table(TABLE_CUSTOMER_VIEWS)

    # Duplicate data
    seen_data=first_occurance.join(
        unique_data,
        on=["customer_id","updated_at"],
        how="inner"
    ).select(["customer_id","first_name","last_name","email","phone","city","region","customer_segment","registration_date","updated_at"])

    # Unique/Silver data
    silver_df = unique_data.join(
        first_occurance,
        on=["customer_id", "updated_at"],
        how="left_anti"
    ).select(["customer_id","first_name","last_name","email","phone","city","region","customer_segment","registration_date","updated_at"])

    silver_df.write.mode("append").format("delta").saveAsTable(TABLE_SILVER_CUSTOMER)
    valid_data_count=silver_df.count()


    # Load duplicate data
    quarantine_data=(duplicate_data.unionByName(
            seen_data
        )) .withColumn(
            "failure_reason",lit("duplicate record")).withColumn("quarantined_at",current_timestamp())
    
    quarantine_data.write.mode("append").format("delta").saveAsTable(TABLE_QUARANTINE_CUSTOMER)

    duplicate_records=quarantine_data.count()

    # Load customer viewed

    viewd_data=silver_df.withColumn("viewed_at",current_timestamp()).select(["customer_id","updated_at","viewed_at"])
    
    viewd_data.write.mode("append").format("delta").saveAsTable(TABLE_CUSTOMER_VIEWS)



    # Updated batch details to metric table 
    metric = [
        Row(
            metric_time=datetime.now(),
            batch_id=batch_id,
            query_name="customer_silver",  
            total_records=batch_df.count(),
            valid_records=valid_data_count,
            invalid_records=batch_df.count()  - valid_data_count,
            duplicate_records=duplicate_records
        )
    ]

    metric_df = spark.createDataFrame(metric)

    metric_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(CUSTOMER_METRICS_TABLE)




In [0]:
query=(
    customer_df
    .writeStream
    .trigger(availableNow=True)
    .foreachBatch(process_dataframe)
    .option("checkpointLocation",PATH_CUSTOMER_CHECKPOINTLOCATION_SILVER)
    .start()
)
query.awaitTermination()

In [0]:

import json
from pyspark.sql import Row
from datetime import datetime

metrics = []

for p in query.recentProgress:

    progress = json.loads(p.json)

    source = progress["sources"][0]

    metrics.append(
        Row(
            metric_time=datetime.now(),
            query_name="customer_silver",
            batch_id=int(progress["batchId"]),
            input_rows=int(source.get("numInputRows", 0)),
            input_rows_per_second=float(source.get("inputRowsPerSecond", 0.0)),
            processed_rows_per_second=float(source.get("processedRowsPerSecond", 0.0)),
            processing_time_ms=int(
                progress.get("durationMs", {}).get("triggerExecution", 0)
            )
        )
    )

if metrics:
    metrics_df = spark.createDataFrame(metrics)

    metrics_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(TABLE_METRIC)
        

In [0]:
%sql
select * from customer_360.silver.customers

In [0]:
%sql
select * from customer_360.bronze.customer_views;

In [0]:
%sql
select * from customer_360.quarantine.customers;

In [0]:
%sql
select * from customer_360.raw.customer_silver_metrics;

In [0]:
%sql
select * from customer_360.raw.stream_metrics;

In [0]:
%sql 
-- drop table if exists customer_360.bronze.customer_views;
-- drop table if exists customer_360.quarantine.customers;
-- drop table if exists customer_360.silver.customers;
-- drop table if exists customer_360.raw.customer_silver_metrics;